# What the framework promises, and what each promise costs

The framework stands between two contracts.
To client code it promises what the summary of [client-authors.ipynb](client-authors.ipynb) lists: records never change, a value lives while something keeps it, a reader never sees a value change, and so on.
To bindings it promises when and how it calls them, as listed in section 7 of [workflow-authors.ipynb](workflow-authors.ipynb).

This notebook does not explain how the workspace works inside.
It shows the promises at work on one example, seen from both sides at once, and then asks for each promise what keeping it costs: in the *local workspace*, the workspace in the user's process that `local()` makes, which exists, and on the service, which is designed but not built.
How the local workspace keeps them is described in the docstring of `ess.apps.workspace`.

## 1. One accumulator, seen from both sides

The driver of `IOFQ_MULTI` and the function of `FORWARD`, from `toy_sans.py`, are wrapped so that every call into them is noted with its time.
The calls of the client are noted when they return, in the left column; the calls into the bindings in the right column.
`FORWARD` is slowed to 300 ms, so that waiting for it shows.

In [ ]:
import tempfile
import threading
import time
from pathlib import Path

from ess.apps import Client, Template, Workspace, file
from ess.apps.testing import FakeFiles, FakeStore
from ess.spec import Driver, DriverBinding, function_binding
from toy_sans import (
    CAN_RUNS,
    FORWARD,
    IOFQ_MULTI,
    SAMPLE_RUNS,
    ForwardOutputs,
    ForwardParams,
    IofQOutputs,
    forward,
    multi,
    single,
)

folder = tempfile.TemporaryDirectory()


class Trace:
    def __init__(self) -> None:
        self._start = time.monotonic()
        self._lines: list[tuple[float, str, str]] = []
        self._lock = threading.Lock()

    def note(self, side: str, what: str) -> None:
        with self._lock:
            self._lines.append((time.monotonic() - self._start, side, what))

    def show(self) -> None:
        print(f'{"time":>8}  {"client":34}  binding')
        for t, side, what in self._lines:
            indent = '' if side == 'client' else ' ' * 36
            print(f'{1000 * t:5.0f} ms  {indent}{what}')


class TracedDriver:
    def __init__(self, driver: Driver[IofQOutputs], trace: Trace) -> None:
        self._driver, self._trace, self._n = driver, trace, 0

    def push(self, table: str, rows: list) -> None:
        self._n += 1
        self._trace.note('binding', f'push {self._n} into {table} starts')
        self._driver.push(table, rows)

    def outputs(self) -> IofQOutputs:
        self._trace.note('binding', f'outputs() of state {self._n}')
        return self._driver.outputs()


def traced_client(trace: Trace) -> Client:
    files = FakeFiles(proposal='p1', directory=Path(folder.name))
    for runs, role in ((SAMPLE_RUNS, 'sample'), (CAN_RUNS, 'background')):
        for run, data in runs.items():
            files.measure(run, data, role=role)

    def make_driver(fixed: dict) -> TracedDriver:
        trace.note('binding', 'make_driver()')
        return TracedDriver(multi.make_driver(fixed), trace)

    def slow_forward(params: ForwardParams) -> ForwardOutputs:
        trace.note('binding', 'FORWARD starts')
        time.sleep(0.3)
        trace.note('binding', 'FORWARD returns')
        return forward(params)

    bind = [
        single,
        DriverBinding(IOFQ_MULTI, make_driver),
        function_binding(FORWARD, slow_forward),
    ]
    workspace = Workspace(proposal='p1', files=files, bind=bind, store=FakeStore())
    return Client(workspace, submitter='alice', on_close=workspace.close)

The client code: four pushes, a read of state 2, a freeze of state 4, and a request that reads the record of the freeze.

In [ ]:
trace = Trace()
client = traced_client(trace)


def note(what: str) -> None:
    trace.note('client', what)


total = client.accumulator(
    Template(
        IOFQ_MULTI, params={'transmission': {'value': 0.8}}, blanks=('sample', 'can')
    )
)
note('accumulator() returns')
total.push(can=[file(run=110)])
note('push 1 returns')
total.push(sample=[file(run=101)])
note('push 2 returns')
state_2 = client.output(total, 'iofq')
note('output() returns: state 2, copied')
total.push(sample=[file(run=102)])
note('push 3 returns')
total.push(sample=[file(run=103)])
note('push 4 returns')
frozen = client.freeze(total)
note('freeze() returns: the record of state 4')
forward_4 = client.submit(FORWARD, params={'curve': frozen.ref('iofq')})
note('submit(FORWARD) returns')
client.wait([frozen, forward_4])
note('wait() returns')
trace.show()
client.close()

Reading the trace from top to bottom:

- **Every call that starts work returns at once.** `accumulator`, `push`, `freeze`, and `submit` return once the workspace has logged them. The binding works later, on the workspace's workers.
- **A freeze does not wait for pushes to be added.** `freeze()` returns the record of state 4 before pushes 3 and 4 are added. FORWARD, which references that record, waits for state 4, as for any pending record.
- **`client.output` waits for its state**, here state 2, and copies it, so the client holds a value that later pushes cannot change.
- **Push n+1 waits for the readers of state n**: the reads by `client.output`, until they have copied, and a freeze, while it is checked. Push 3 starts after state 2 is copied, since the driver may add push 3 in place.
- **A request never holds back a push.** It reads an accumulator only through the record of its freeze, and no push follows a freeze. FORWARD reads the outputs of state 4 without a copy.
- **`outputs()` is called once per state that is read or frozen.** State 1, with no sample row, and state 3 are never read.
- **`make_driver` is called once**, and the pushes come one at a time, in the order logged.

The client sees the same order as if every call ran after the one before it; the binding sees a driver that only moves forward.

One more promise, and its limit: cancelling ends a record at once, but a workflow that has started runs on.

In [ ]:
trace = Trace()
client = traced_client(trace)
reference = client.compute(
    IOFQ_MULTI,
    params={
        'sample': [file(run=101)],
        'can': [file(run=110)],
        'transmission': {'value': 0.8},
    },
)
slow = client.submit(FORWARD, params={'curve': reference.ref('iofq')})
time.sleep(0.05)
client.cancel(slow)
note(f'cancel() returns: the record is {client.status(slow).value}')
time.sleep(0.4)
trace.show()

## 2. The promises, and what keeping them costs

The first two columns are the contract; the last two are the price.
"This process" is the local workspace.
"Service" is what DMSC, the ESS data centre, hosts for batch and automatic reduction, which is designed only.
It runs one workspace for each proposal, and workspaces share values only through the store.
Letters and numbers such as D3 name stories in [system-stories.md](../system-stories.md).

| Promise | Relied on by | In this process | On the service |
|---|---|---|---|
| A record never changes; only its status does. | clients, provenance | records are entries in an append-only log (ADR 0004) | the same append-only lists, stored durably in a form the service chooses (ADR 0004); a new version must read an old one's records (H2) |
| A refused call submits nothing, and names the field. | apps (forms), batches | every request of a call is checked before any is logged, without reading data: each file resolved, each parameter that holds no reference to an output validated with the local path of each file it holds, each reference to an output checked for its fit to the type of its field | the same; the service needs every spec it accepts |
| A workflow gets each value as the type its parameter declares, whether written inline, a file, or a reference to an output. | bindings | the workspace validates each value written inline and resolves each file at submission, reads the data of each reference to an output when the workflow runs, and validates the stored form with the params model, which validates each group that holds a reference, without running the model validators of the params model | the data must be read where the workflow runs |
| A request runs once the records it references have completed. | chains | the only scheduling there is | a request must reach the data it references where it runs |
| The rows of a table are distinct, by their stored form. | accumulators: a run is never summed twice | a push is compared with the table as the records store it when it is logged: a file by its identity, a reference to an output by its record and output | the same |
| A value lives while something keeps it, and only then; work that nothing keeps is cancelled. | clients (memory), the service (bounded memory) | the workspace counts keepers per value | how the service bounds the memory of a workspace, and when it ends an idle workspace, are open (ADR 0002, scipp/essapps#27) |
| Persisted values outlive the workspace; history goes as a whole once the proposal is idle. | batch and automatic reduction | a store passed to `local`; history ends with the process | a store per proposal; when and by whom values are dropped from it before the history is, to free disk, is open (ADR 0002, H1, H3) |
| A reader never sees a value change: push n+1 waits for the readers of state n. | clients that read states; drivers that add in place | a wait on one driver in one process | a read copies where the driver is; a request that reads the record of a freeze must run where the driver is, or get a copy (ADR 0003; README open question 5) |
| A read or a freeze is of the state after the pushes logged when it is called, and a freeze does not wait for them to be added. | controllers in loops | the state is the number of pushes logged | the same |
| A call through a stage, and a freeze of an accumulator, make the record of their plain request. | provenance, rerunning | the `sciline.Stage` and the driver keep the symmetry; `local()` never restarts, and a workspace started on a file log reruns a call through a stage as its plain request, and fails the record of a freeze | the calls through a stage must run where the binding's stage is held |
| Where a request runs changes neither its record nor its outputs. | everybody | one place | the same software everywhere; records name no software versions (F2) |
| A proposal reads only its own files and records. | DMSC (G5) | a workspace holds the history of one proposal, so a reference can name only that proposal's records; the workspace checks each file against its proposal | the same check behind authentication; a modified client cannot get around it |
| Cancelling ends a record at once. | batches restarted after a mistake (D3) | a running workflow is not interrupted | its worker stays busy until it returns, and the resubmitted batch waits (D3 gap) |

To bindings the framework promises the calls of the table in section 7 of [workflow-authors.ipynb](workflow-authors.ipynb).
Their cost: one driver takes one push at a time, on one worker.
Spreading one accumulator over several workers would need a merge of two drivers, which the driver protocol does not offer (README open question 1).

## 3. Which promises are proven only in this process

Every promise above has one implementation, the local workspace.
The ones whose service column is open are where the contract may still have to change:

- **Where a driver lives.** "A reader never sees a value change" costs one wait in one process. On the service, readers of a state on other workers need a copy, or must run where the driver is.
- **Memory.** "A value lives while something keeps it" bounds memory in the user's process by what the user has not released. How the service bounds the memory of a workspace, when it ends an idle workspace, whether the users of one proposal share the values in memory or the service separates or bounds them per user, and where a workspace's values are held when its work spreads over nodes, are open (ADR 0002, scipp/essapps#27).
- **Copies.** In the user's process, `client.output` of a record the workspace keeps returns the value itself, not a copy. Notebooks with large results profit, and modifying the value in place is unguarded (README open question 7). On the service every read is a copy, which `select=` limits to a slice (B4).
- **Cancelling.** A cancelled workflow that has started keeps its worker.
- **Software versions.** Placement and rerunning a record assume the same software, which records do not name.

The rest, the log, the keepers, and the waits, are implementation details as long as these promises hold.

In [ ]:
client.close()
folder.cleanup()